# Офлайн-проверка инференса

Тетрадь проверяет связку «предобработка → модель» без поднятия сервиса: берёт реального
клиента из `data/train_ver2.csv`, готовит признаки общим модулем `preprocessing.py`
(тем же, что использует сервис `app1.py`) и скорит канонической моделью
`fastapi/saved_model.pkl`.

Предусловия: прогнаны `loader.ipynb` (данные) и `modeling.ipynb` (артефакты в `fastapi/`).
Автоматизированный вариант этой проверки — `tests/` (`pytest`).

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from preprocessing import PreprocessingParams, load_personal_recs, prepare_features

ARTIFACTS_DIR = Path('fastapi')
MODEL_PATH = ARTIFACTS_DIR / 'saved_model.pkl'
PARAMS_PATH = ARTIFACTS_DIR / 'preprocessing_params.json'
PERSONAL_RECS_PATH = ARTIFACTS_DIR / 'personal_als.parquet'

In [ ]:
# Канонический экспорт модели (CODE_REVIEW §P2.20): один файл для сервиса и офлайна.
assert MODEL_PATH.exists(), (
    f'{MODEL_PATH} нет — прогоните modeling.ipynb (ячейка экспорта модели)')
model = joblib.load(MODEL_PATH)
params = PreprocessingParams.from_json(PARAMS_PATH)
personal_recs = load_personal_recs(PERSONAL_RECS_PATH)

print(f'Модель: {MODEL_PATH} (классы: {list(model.classes_)})')
print(f'Параметры: {params.source} (сохранены {params.created_at})')
recs_info = f'{len(personal_recs)} клиентов' if personal_recs is not None else 'нет'
print(f'Персональные рекомендации: {recs_info}')
print(f'label_map: {params.label_map}')

In [ ]:
# Реальный клиент из датасета — целиком одной строкой, а не Frankenstein-профиль
# из несовместимых значений разных строк (CODE_REVIEW §5.3).
DATA_PATH = Path('data/train_ver2.csv')
assert DATA_PATH.exists(), f'{DATA_PATH} нет — прогоните loader.ipynb'
client = pd.read_csv(DATA_PATH, nrows=5000).sample(1, random_state=42).iloc[0].to_dict()
print(f"ncodpers={client['ncodpers']}, age={client['age']}, segmento={client['segmento']}")

In [ ]:
# Та же функция, что внутри POST /predict: расхождение обучения и сервиса
# исключено по построению (CODE_REVIEW §P2.14).
features = prepare_features(client, params, personal_recs)
print(f'Признаков: {features.shape[1]}')
print(features.iloc[0][['age_interval', 'total_products', 'recommended_product_id',
                        'mean_renta_by_pais_residencia', 'renta_vs_country_mean']])

In [ ]:
probabilities = model.predict_proba(features)[0]
order = np.argsort(-probabilities, kind='stable')
classes = [int(cls) for cls in model.classes_]
print(f"Клиент {client['ncodpers']}:")
for rank in order[:3]:
    code = classes[rank]
    print(f'  {code:>2} {params.label_map.get(code, "?"):<22} p={probabilities[rank]:.3f}')

## Что дальше

- Совпадение с сервисом: поднимите `uvicorn app1:app --port 8079` и отправьте этот же
  `client` в `POST /predict` — ответ должен совпасть с топ-1 выше (автоматически это
  проверяет `tests/test_api.py::test_predict_matches_offline`).
- Нагрузочный прогон сервиса — `test.ipynb`.
- Версия модели (run id, дата, метрики) — `fastapi/model_version.json`.